In [4]:
pip install gradio ultralytics torch opencv-python numpy

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gradio as gr
from ultralytics import YOLO
import cv2
import os

# Load your model 
model = YOLO("yolov8n-pose.pt")

cancel_flag = False

def stop_processing():
    global cancel_flag
    cancel_flag = True

def process_image(img):
    results = model(img)
    return results[0].plot()

def process_video(video):
    global cancel_flag
    cancel_flag = False

    if video is None:
        return None

    # === CONFIG ===
    predict_every_n_frames = 1       
    imgsz = 640                      
    
    current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
    final_path = os.path.join(current_dir, "final_output_video.mp4")

    # === Open video ===
    cap = cv2.VideoCapture(video)
    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    # === Video writer (CHANGED TO WEB-FRIENDLY avc1) ===
    # Using 'avc1' directly writes an H.264 MP4 that browsers can play natively!
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    out = cv2.VideoWriter(final_path, fourcc, fps, (width, height))

    frame_idx = 0

    while cap.isOpened():
        if cancel_flag:
            break

        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_idx % predict_every_n_frames == 0:
            results = model.predict(source=frame, imgsz=imgsz, verbose=False)
            annotated_frame = results[0].plot()
        else:
            annotated_frame = frame

        out.write(annotated_frame)
        frame_idx += 1

    cap.release()
    out.release()

    # If the user stopped it instantly or video was empty
    if frame_idx == 0:
        if os.path.exists(final_path):
            os.remove(final_path)
        return None

    print(f"✅ Done! Web-ready video saved directly to: {final_path}")
    
    # Return the file directly from the directory to Gradio
    return final_path


css = """
#title{ text-align:center; font-size:45px; font-weight:bold; color:#1e3a8a; }
#subtitle{ text-align:center; font-size:18px; margin-bottom:20px; }
#team{ text-align:center; margin-bottom:25px; font-size:16px; }
footer{ text-align:center; margin-top:30px; font-size:14px; color:#475569; }
.gr-button.primary { background:#2563eb !important; color:white !important; font-weight:bold; border-radius:8px; }
.gr-button.stop-btn { background:#dc2626 !important; color:white !important; font-weight:bold; border-radius:8px; }
"""

with gr.Blocks(css=css) as demo:
    gr.Markdown("<div id='title'>Human Pose Estimation with YOLOv8</div>")
    gr.Markdown("<div id='subtitle'>Upload an image or video to detect human body keypoints using AI.</div>")
    gr.Markdown("<div id='team'>Team Members: Write your names here</div>")

    with gr.Tabs():
        with gr.Tab("Image Detection"):
            img_input = gr.Image(type="numpy", label="Upload Image")
            img_output = gr.Image(label="Result")
            img_btn = gr.Button("Run Detection", elem_classes="primary")
            img_btn.click(process_image, img_input, img_output)

        with gr.Tab("Video Detection"):
            vid_input = gr.Video(label="Upload Video")
            vid_output = gr.Video(label="Processed Video", format="mp4") 
            
            with gr.Row():
                vid_btn = gr.Button("Run Detection", elem_classes="primary")
                stop_btn = gr.Button("Stop Detection (Kill Switch)", elem_classes="stop-btn")

            vid_btn.click(process_video, vid_input, vid_output)
            stop_btn.click(stop_processing)
            
    gr.Markdown("<footer>AI Pose Detection Project using YOLOv8. Enjoy exploring human poses! 😊</footer>")

demo.queue()
demo.launch(share=True)

C:\Users\alkin\AppData\Local\Temp\ipykernel_27460\2265316423.py:87: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=css) as demo:


* Running on local URL:  http://127.0.0.1:7865

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


✅ Done! Web-ready video saved directly to: d:\computer vision\project\ComputerVision_FP-main\ComputerVision_FP-main\final_output_video.mp4
✅ Done! Web-ready video saved directly to: d:\computer vision\project\ComputerVision_FP-main\ComputerVision_FP-main\final_output_video.mp4
